# CEG-WM 裁剪瓶颈：8 对样本最小诊断

仅运行固定前 8 个正式源的重生成与四路径评分，**不运行正式 worker、不训练、不重新校准、不搜索变换**。
这是新一轮机制诊断，不保证重现历史 PNG 字节或历史分数。

运行前：选择 GPU 运行时（建议 A100 和高 RAM；原管线完整驻留 GPU），在 Colab Secrets 中提供
`HF_TOKEN`、`CEG_WM_ROOT_KEY` 并允许本 notebook 访问。HF 账户需已有 SD3.5 模型访问权限。
然后运行全部单元。首次会下载模型。代码只使用固定历史提交，诊断脚本已内嵌，无须另行上传 Python 文件。

结果保存在 Drive `CEG-WM/crop-bottleneck-v1/attempt01/`。中断后使用相同 RUN_ID 再次运行全部单元；
已提交的成对 PNG 和已完成评分会校验哈希并跳过，失败和中断尝试保留。
换依赖、GPU 型号、模型版本或诊断脚本时会拒绝混合续跑；要另设 RUN_ID。

**本入口已做本地工程检查，尚未执行真实 Colab/GPU 实验。不要据此宣称水印或 Oracle 已通过。**


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, sys, json, re
RUN_ID = 'attempt01'
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID)
output = pathlib.Path('/content/drive/MyDrive/CEG-WM/crop-bottleneck-v1') / RUN_ID
checkout = pathlib.Path('/content/cegwm-crop-v1-source')
runtime = pathlib.Path('/content/cegwm-crop-v1-runtime')
script = pathlib.Path('/content/run_crop_diagnostic.py')
expected = 'd9ef4adb4f8424f1102f577bb76347fbba112bd9'
child_env = dict(os.environ)
for name in ('HF_TOKEN', 'CEG_WM_ROOT_KEY'):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f'Missing Colab secret: {name}')
    child_env[name] = value
child_env['HF_HOME'] = '/content/cegwm-crop-v1-hf'
child_env['HF_HUB_CACHE'] = child_env['HF_HOME'] + '/hub'
child_env['TOKENIZERS_PARALLELISM'] = 'false'
child_env['PYTHONUNBUFFERED'] = '1'
# Check in a child: pip changes below cannot leave stale imported modules in the kernel.
subprocess.run([sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(), "Select a Colab GPU runtime"; '
    'print({"gpu":torch.cuda.get_device_name(0),"vram_GiB":round(torch.cuda.get_device_properties(0).total_memory/2**30,1)})'], check=True)


In [ ]:
# Preserve Colab's matched CUDA torch/torchvision. These are preparation pins,
# not a claim that this exact full-model environment has already passed.
dependencies = ['diffusers==0.39.0', 'transformers==4.57.6', 'accelerate==1.1.1',
                'sentencepiece==0.2.0', 'protobuf>=3.20']
subprocess.run([sys.executable, '-m', 'pip', 'install', *dependencies], check=True)
if not checkout.exists():
    subprocess.run(['git', 'clone', 'https://github.com/RICHAAARC/CEG-WM.git', str(checkout)], check=True)
actual = subprocess.run(['git', '-C', str(checkout), 'rev-parse', 'HEAD'],
                        capture_output=True, text=True, check=True).stdout.strip()
if actual != expected:
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', expected], check=True)
child_env['PYTHONPATH'] = str(checkout / 'src') + os.pathsep + str(checkout)
subprocess.run([sys.executable, '-c',
    'from cegwm.shared.keys import normalize_detection_key,public_key_digest; '
    'from cegwm.protocol.content_chain import CONTENT_CHAIN_PUBLIC_KEY_DIGEST; '
    'import os; assert public_key_digest(normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"])) == CONTENT_CHAIN_PUBLIC_KEY_DIGEST, "Detection key identity differs"; '
    'from diffusers import StableDiffusion3Pipeline; '
    'from transformers import AutoModel, AutoImageProcessor; '
    'print("Imports and key identity OK; no models loaded")'], env=child_env, check=True)


In [ ]:
# Self-contained diagnostic implementation, identical to delivered Python file.
script.write_text('"""Eight-source crop diagnosis. Reuses the pinned paper writer/reader unchanged."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom dataclasses import asdict\nimport hashlib\nfrom importlib.metadata import version\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport uuid\n\nEXACT = "d9ef4adb4f8424f1102f577bb76347fbba112bd9"\nROUTES = ("clean", "crop_direct", "predicted_h", "oracle_h")\nROLES = ("positive", "negative")\nMODELS = ("stabilityai/stable-diffusion-3.5-medium", "facebook/dinov2-small")\nTAU = 1.5545224858704942\nTERMINAL = ("SCORED", "NO_VALID_H")\n\n\ndef public_error(error):\n    message = "".join(traceback.format_exception(error))\n    for name in ("HF_TOKEN", "CEG_WM_ROOT_KEY"):\n        secret = os.environ.get(name)\n        if secret:\n            message = message.replace(secret, "[REDACTED]")\n    return {"type": type(error).__name__, "traceback_redacted": message}\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with tmp.open("x", encoding="utf-8") as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(tmp, path)\n\n\ndef bind(path, value):\n    """Reject mixed protocol/assets/environment instead of silently resuming."""\n    if Path(path).exists():\n        if read(path) != value:\n            raise RuntimeError(f"Resume identity differs: {Path(path).name}; use a new RUN_ID")\n    else:\n        write(path, value)\n\n\ndef save_image(path, image):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    image.save(tmp, format="PNG")\n    os.replace(tmp, path)\n    return {"path": str(path), "sha256": digest(path)}\n\n\ndef verify_file(record):\n    if digest(record["path"]) != record["sha256"]:\n        raise RuntimeError("Saved artifact hash differs; refusing mixed resume")\n\n\ndef crop_geometry(size=512):\n    """Physical pixel centers and PIL-edge sampler coordinates, separately."""\n    import numpy as np\n    extent = round(size * math.sqrt(0.80))\n    left = (size - extent) // 2\n    scale = extent / size\n    # PIL resize: original center = left + (observed_center+.5)*scale-.5.\n    center = np.array([[scale, 0., left + .5 * scale - .5],\n                       [0., scale, left + .5 * scale - .5], [0., 0., 1.]])\n    edge = np.array([[scale, 0., left], [0., scale, left], [0., 0., 1.]])\n    norm = np.array([[2 / (size - 1), 0., -1.],\n                     [0., 2 / (size - 1), -1.], [0., 0., 1.]])\n    return {\n        "crop_box": [left, left, left + extent, left + extent],\n        "retained_area_fraction": (extent / size) ** 2,\n        "physical_h_observed_to_canonical": (norm @ center @ np.linalg.inv(norm)).tolist(),\n        # Frozen renderer passes pixel coefficients straight to PIL. Its sampler\n        # evaluates pixel-edge coordinates; this H yields exactly x -> (x-left)/s.\n        "sampler_h_observed_to_canonical": (norm @ edge @ np.linalg.inv(norm)).tolist(),\n        "oracle_policy": "analytic_crop_inverse_black_bilinear_no_original_pixels",\n    }\n\n\ndef corner_error(predicted, truth):\n    import numpy as np\n    q = np.array([[-1., -1., 1.], [1., -1., 1.], [1., 1., 1.], [-1., 1., 1.]]).T\n    a, b = np.asarray(predicted) @ q, np.asarray(truth) @ q\n    a, b = a[:2] / a[2], b[:2] / b[2]\n    return float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=0))) * 511 / 2)\n\n\ndef summarize(root, roster):\n    rows = []\n    for unit in roster:\n        for role in ROLES:\n            for route in ROUTES:\n                path = root / "units" / unit["unit_id"] / role / route / "result.json"\n                row = read(path) if path.exists() else {"status": "PENDING"}\n                rows.append({**row, "unit_id": unit["unit_id"], "role": role, "route": route})\n    counts = {}\n    for role in ROLES:\n        for route in ROUTES:\n            subset = [r for r in rows if r["role"] == role and r["route"] == route]\n            counts[f"{role}/{route}"] = {\n                "planned": len(roster),\n                "scored": sum(r["status"] == "SCORED" for r in subset),\n                "above_historical_reference_tau": sum(r.get("above_reference_tau", False) for r in subset),\n                "no_valid_h": sum(r["status"] == "NO_VALID_H" for r in subset),\n                "operational_error": sum(r["status"] == "OPERATIONAL_ERROR" for r in subset),\n                "pending": sum(r["status"] == "PENDING" for r in subset),\n            }\n    complete = all(r["status"] in TERMINAL for r in rows)\n    result = {"status": "COMPLETE_DIAGNOSTIC" if complete else "INCOMPLETE",\n              "planned_source_pairs": 8, "planned_route_records": 64,\n              "claim_ceiling": "engineering_diagnosis_only_not_formal_TPR_or_FPR",\n              "tau_role": "historical_reference_only_no_recalibration",\n              "counts": counts, "records": rows}\n    write(root / "summary.json", result)\n    csv_path = root / "readout.csv"\n    tmp = csv_path.with_suffix(".csv.tmp")\n    with tmp.open("w", encoding="utf-8", newline="") as stream:\n        columns = ("unit_id", "role", "route", "status", "score", "margin_to_reference_tau",\n                   "registered_weighted_joint", "wrong_key_max", "lf_registered", "hf_registered",\n                   "corner_rmse_physical_pixels")\n        writer = csv.DictWriter(stream, fieldnames=columns)\n        writer.writeheader()\n        for row in rows:\n            stats = row.get("statistic", {})\n            branches = row.get("branches", {})\n            writer.writerow({**{key: row.get(key) for key in columns}, "score": stats.get("value"),\n                             "registered_weighted_joint": stats.get("registered_weighted_joint"),\n                             "wrong_key_max": stats.get("wrong_key_max"),\n                             "lf_registered": branches.get("lf", {}).get("registered"),\n                             "hf_registered": branches.get("hf", {}).get("registered")})\n    os.replace(tmp, csv_path)\n    return result\n\n\ndef prepare_assets(root):\n    """Pin current public model revisions for THIS rerun, not claim old weights."""\n    from huggingface_hub import HfApi, snapshot_download\n    path = root / "model_revisions.json"\n    token = os.environ["HF_TOKEN"]\n    if path.exists():\n        revisions = read(path)\n    else:\n        revisions = {name: HfApi().model_info(name, token=token).sha for name in MODELS}\n        write(path, revisions)\n    cache = Path(os.environ["HF_HUB_CACHE"])\n    for name in MODELS:\n        patterns = (["model_index.json", "scheduler/*", "text_encoder/*", "text_encoder_2/*",\n                     "text_encoder_3/*", "tokenizer/*", "tokenizer_2/*", "tokenizer_3/*",\n                     "transformer/*", "vae/*"] if name == MODELS[0] else ["*.json", "*.safetensors"])\n        snapshot_download(name, revision=revisions[name], token=token, cache_dir=str(cache),\n                          allow_patterns=patterns,\n                          ignore_patterns=["*.bin", "*.h5", "*.msgpack", "*.onnx", "*.ot"])\n        # Isolated cache only. The unchanged historical loader requests main.\n        # Resolve it to the explicitly downloaded SHA before the OFFLINE child.\n        ref = cache / ("models--" + name.replace("/", "--")) / "refs" / "main"\n        ref.parent.mkdir(parents=True, exist_ok=True)\n        ref.write_text(revisions[name], encoding="ascii")\n\n\ndef score_image(runtime, image):\n    from cegwm.runtime.blind_scoring_v2 import score_branches_v2\n    from cegwm.method.blind_detection import statistic_from_weighted_scores\n    branches, mode = score_branches_v2(image, runtime["key"], runtime["assets"], reuse_observation=True)\n    score = statistic_from_weighted_scores(branches["weighted_joint"])\n    return {"branches": branches, "statistic": asdict(score), "observation_mode": mode,\n            "above_reference_tau": score.value > TAU, "margin_to_reference_tau": score.value - TAU}\n\n\ndef execute(root, repo, runtime_root):\n    import torch\n    from PIL import Image\n    from experiments.run_paper_main_worker_v2 import _build_runtime, _main_pair\n    from cegwm.formal_experiment_v2 import load_formal_config, expand_rosters, apply_attack\n    from cegwm.runtime.blind_detection import _geometry_disposition, _raw_h\n    from cegwm.geometry_v7.r1b import rectify_attacked_rgb\n    from cegwm.shared.keys import normalize_detection_key, public_key_digest\n\n    actual = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()\n    if actual != EXACT:\n        raise RuntimeError("Historical code revision differs")\n    if subprocess.check_output(["git", "-C", str(repo), "diff", "HEAD", "--"], text=True):\n        raise RuntimeError("Historical tracked source is modified")\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU required; no model was loaded")\n    config = load_formal_config(repo / "configs/paper_experiment/formal_experiment_v2.json")\n    roster = [asdict(u) for u in expand_rosters(repo, config)["formal_evaluation_pairs"][:8]]\n    key_digest = public_key_digest(normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"]))\n    protocol = {"schema": "crop_bottleneck_v1", "source_exact": EXACT, "roster": roster,\n                "script_sha256": digest(__file__), "roles": list(ROLES), "routes": list(ROUTES),\n                "reference_tau": TAU, "key_digest": key_digest, "crop": crop_geometry(),\n                "negative_role": "same_seed_primary_null_without_syncseal",\n                "prediction_policy": "single_continuous_H_original_renderer_no_search",\n                "C_is_post_only": True, "original_paper_decision_reconstructible_from_B_C": True}\n    bind(root / "protocol.json", protocol)\n    summarize(root, roster)\n    versions = {name: version(name) for name in\n                ("torch", "torchvision", "diffusers", "transformers", "accelerate", "numpy", "Pillow", "huggingface_hub", "safetensors", "sentencepiece")}\n    env = {"packages": versions, "python": platform.python_version(), "cuda": torch.version.cuda,\n           "gpu": torch.cuda.get_device_name(0), "model_revisions": read(root / "model_revisions.json")}\n    bind(root / "environment.json", env)\n    if summarize(root, roster)["status"] == "COMPLETE_DIAGNOSTIC":\n        for unit in roster:\n            unit_root = root / "units" / unit["unit_id"]\n            pair = read(unit_root / "pair.json")\n            for artifact in pair.values():\n                verify_file(artifact)\n            for role in ROLES:\n                for route in ROUTES:\n                    row = read(unit_root / role / route / "result.json")\n                    for name in ("image", "valid_support"):\n                        if name in row:\n                            verify_file(row[name])\n        print("Already complete; no generation or scoring repeated.", flush=True)\n        return\n    runtime = _build_runtime(runtime_root)\n    checkpoints = sorted(runtime_root.glob("syncseal-*/syncmodel.jit.pt"), key=lambda p: p.stat().st_mtime)\n    bind(root / "syncseal_identity.json", {"sha256": digest(checkpoints[-1])})\n    # Copy of the exact executable script for provenance; never store credentials.\n    saved_script = root / "run_crop_diagnostic.py"\n    if not saved_script.exists():\n        saved_script.write_bytes(Path(__file__).read_bytes())\n    truth = crop_geometry()\n    for unit in roster:\n        unit_root = root / "units" / unit["unit_id"]\n        pair_path = unit_root / "pair.json"\n        print(f"SOURCE {unit[\'unit_id\']}", flush=True)\n        if not pair_path.exists():\n            attempt = unit_root / "generation_attempts" / uuid.uuid4().hex\n            write(attempt / "state.json", {"status": "STARTED", "time": time.time()})\n            try:\n                clean, marked = _main_pair(runtime, unit["prompt"], unit["seed"])\n                pair = {"negative": save_image(attempt / "clean.png", clean),\n                        "positive": save_image(attempt / "watermarked.png", marked)}\n                write(attempt / "state.json", {"status": "COMPLETE", "time": time.time()})\n                write(pair_path, pair)\n            except Exception as error:\n                write(attempt / "state.json", {"status": "OPERATIONAL_ERROR", **public_error(error)})\n                for role in ROLES:\n                    for route in ROUTES:\n                        write(unit_root / role / route / "result.json",\n                              {"status": "OPERATIONAL_ERROR", "stage": "generation", **public_error(error)})\n                summarize(root, roster)\n                raise\n        pair = read(pair_path)\n        for role in ROLES:\n            verify_file(pair[role])\n            with Image.open(pair[role]["path"]) as source:\n                original = source.convert("RGB").copy()\n            attacked = apply_attack(original, "center_crop_80_restore")\n            for route in ROUTES:\n                folder = unit_root / role / route\n                result_path = folder / "result.json"\n                if result_path.exists() and read(result_path)["status"] in TERMINAL:\n                    old = read(result_path)\n                    for name in ("image", "valid_support"):\n                        if name in old:\n                            verify_file(old[name])\n                    continue\n                attempt = folder / "attempts" / uuid.uuid4().hex\n                write(attempt / "state.json", {"status": "STARTED", "time": time.time()})\n                try:\n                    metadata = {}\n                    if route == "clean":\n                        image = original\n                    elif route == "crop_direct":\n                        image = attacked\n                    else:\n                        if route == "predicted_h":\n                            geometry = runtime["assets"].geometry_backend.detect_geometry_continuous(attacked)\n                            disposition, detail = _geometry_disposition(geometry)\n                            metadata["geometry"] = asdict(geometry)\n                            write(attempt / "geometry.json", metadata)\n                            if disposition == "OPERATIONAL":\n                                raise RuntimeError("SyncSeal operational failure")\n                            try:\n                                if disposition == "INVALID_H":\n                                    raise LookupError("No legal predicted H")\n                                matrix = _raw_h(geometry)\n                            except (LookupError, TypeError, ValueError):\n                                result = {"status": "NO_VALID_H", **metadata}\n                                write(attempt / "state.json", result)\n                                write(result_path, result)\n                                summarize(root, roster)\n                                continue\n                            metadata["corner_rmse_physical_pixels"] = corner_error(\n                                matrix, truth["physical_h_observed_to_canonical"])\n                        else:\n                            matrix = truth["sampler_h_observed_to_canonical"]\n                            metadata["truth"] = truth\n                        metadata["h_passed_to_frozen_renderer"] = matrix\n                        image = rectify_attacked_rgb(attacked, matrix)\n                        support = rectify_attacked_rgb(Image.new("RGB", (512, 512), "white"), matrix)\n                        metadata["valid_support"] = save_image(attempt / "support.png", support)\n                    metadata["image"] = save_image(attempt / "input.png", image)\n                    result = {"status": "SCORED", **metadata, **score_image(runtime, image)}\n                    write(attempt / "state.json", result)\n                    write(result_path, result)\n                    print(f"  {role}/{route}: margin={result[\'margin_to_reference_tau\']:.5f}", flush=True)\n                except Exception as error:\n                    result = {"status": "OPERATIONAL_ERROR", "stage": route, **public_error(error)}\n                    write(attempt / "state.json", result)\n                    write(result_path, result)\n                    summarize(root, roster)\n                    raise\n                summarize(root, roster)\n    print(json.dumps(summarize(root, roster)["counts"], indent=2), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--repo", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--runtime", required=True, type=Path)\n    parser.add_argument("--prepare-assets", action="store_true")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    sys.path[:0] = [str(args.repo), str(args.repo / "src")]\n    try:\n        if args.prepare_assets:\n            prepare_assets(args.output)\n        else:\n            execute(args.output, args.repo, args.runtime)\n    except Exception as error:\n        write(args.output / "last_failure.json", {**public_error(error), "time": time.time()})\n        raise\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
command = [sys.executable, str(script), '--repo', str(checkout), '--output', str(output), '--runtime', str(runtime)]
prepare_env = {k:v for k,v in child_env.items() if k not in ('HF_HUB_OFFLINE','TRANSFORMERS_OFFLINE')}
subprocess.run(command + ['--prepare-assets'], env=prepare_env, check=True)
# Model SHA is frozen in Drive; historical loaders use only the isolated pinned cache.
run_env = dict(child_env, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
completed = subprocess.run(command, env=run_env, cwd=checkout, check=False)
summary_path = output / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps({'status': summary['status'], 'counts': summary['counts']}, indent=2))
print('Output directory:', output)
if completed.returncode:
    raise RuntimeError('Diagnostic stopped. Completed PNGs/scores are retained; inspect the error above and rerun after fixing the cause.')


## 结果解释与停止边界

- A=`clean`，B=`crop_direct`，C=`predicted_h`（仅校正后分数），D=`oracle_h`。
- C 不执行原流程的直读阳性提前返回，以便每张图都获得校正后证据；原流程的判定可由 B/C 重建。
- 负样本是同种子 primary null，无 SyncSeal；与旧正式配对一致。
- `summary.json` 固定保留 8 个源、64 个路径槽位。缺失、无合法 H、运行失败分别记录，不改分母。
- `above_historical_reference_tau` 仅是旧阈值参照，不是重新校准后的 TPR/FPR。
- Oracle 仅使用攻击真值，不读取原图补缺失像素。几何误差用物理像素中心矩阵；
  Oracle 传给原 PIL 渲染器的矩阵另做固定半像素坐标转换，二者均落盘。
- 首先检查 A 的可读性；D 好而 C 差支持同步问题；C/D 都差才进一步检查观测或载体。
  **D 失败不证明信息论意义上的载体消失。** 不自动启动掩膜搜索、传播探针或新方法开发。
- `model_revisions.json` 冻结此次重跑的公开模型 SHA；历史模型版本未能确认，因此本轮不是逐字节历史复现。

完成后提供整个结果目录，至少保留 `protocol.json`、`environment.json`、`model_revisions.json`、
`syncseal_identity.json`、`summary.json` 和 `units/`；不要只保留汇总表。
